In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score

import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
from keras.models import Sequential
from keras.layers import Dense, Flatten, Input, Conv1D, MaxPooling1D, UpSampling1D, Reshape
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.layers import Dropout,BatchNormalization
from keras.regularizers import l2
from keras.optimizers import Adam

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd '/content/drive/My Drive/kddcup'
# !pwd
# !ls

#### load the KDDCup99 dataset and specify column names for readability

In [ ]:
# Load dataset
columns = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent",
           "hot", "num_failed_logins", "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
           "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
           "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
           "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
           "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
           "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label"]

df = pd.read_csv("kddcup.data.gz", sep=",", names=columns, index_col=None)

#### convert the 'label' column to binary (0 for normal, 1 for anomaly).

In [ ]:

df['label'] = df['label'].apply(lambda x: 0 if x == 'normal.' else 1)

# Filter for HTTP traffic
df = df[df["service"] == "http"]
df = df.drop("service", axis=1)

#### label encode any categorical features in the dataset to convert them into numerical format.

In [ ]:
# Encode categorical features
for col in df.columns:
    if df[col].dtype == "object":
        encoded = LabelEncoder()
        encoded.fit(df[col])
        df[col] = encoded.transform(df[col])

#### Here, we separate the normal and anomalous data for use in semi-supervised learning.

In [ ]:
# Split into normal (0) and anomalous (1)
df_normal = df[df['label'] == 0]  # Normal traffic
df_anomalous = df[df['label'] == 1]  # Anomalies

#### We take only the normal data (X_normal) for training and testing.
#### We perform an 80-20 split on the normal data for training and validation.

In [ ]:
# Split normal data into training and test sets
X_normal = df_normal.drop(columns=['label'])
y_normal = df_normal['label']

X_train, X_test = train_test_split(X_normal, test_size=0.2, random_state=42)

#### We remove extreme outliers by clipping values beyond the 99th percentile and apply Min-Max scaling to normalize the features.

In [ ]:

# Clip outliers and apply Min-Max scaling
clip_value = np.percentile(X_train, 99)
X_train_clipped = np.clip(X_train, 0, clip_value)
X_test_clipped = np.clip(X_test, 0, clip_value)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_clipped)
X_test_scaled = scaler.transform(X_test_clipped)

#### Building the Autoencoder Model

In [ ]:
##Insert your model here
enc = Sequential([
    Flatten(input_shape=(40,)),
    Dense(100, activation='selu'),
    Dense(30, activation='selu')
])
dec = Sequential([
    Flatten(input_shape=(30,)),
    Dense(100, activation='selu'),
    Dense(40, activation='sigmoid')  # Match the original input shape
])
autoencoder=Sequential([enc, dec])
autoencoder.compile(loss='binary_crossentropy', optimizer=keras.optimizers.SGD(learning_rate=0.01), metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# Train the autoencoder, replace the 'autoencoder' with the model name you set above
# You can update hyperparameters and add regularization techniques as needed
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
model_checkpoint = ModelCheckpoint(filepath='best_autoencoder.keras', monitor='val_loss', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1)


history = autoencoder.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=32,
                          validation_split=0.2, verbose=1,callbacks=[early_stop, model_checkpoint, reduce_lr])

Epoch 1/50
12372/12381 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0898 - loss: 0.1265
Epoch 1: val_loss improved from inf to 0.06315, saving model to best_autoencoder.keras
12381/12381 ━━━━━━━━━━━━━━━━━━━━ 35s 3ms/step - accuracy: 0.0898 - loss: 0.1265 - val_accuracy: 0.0010 - val_loss: 0.0632 - learning_rate: 0.0100
Epoch 2/50
12374/12381 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.0011 - loss: 0.0614
Epoch 2: val_loss improved from 0.06315 to 0.05820, saving model to best_autoencoder.keras
12381/12381 ━━━━━━━━━━━━━━━━━━━━ 39s 3ms/step - accuracy: 0.0011 - loss: 0.0614 - val_accuracy: 0.0018 - val_loss: 0.0582 - learning_rate: 0.0100
Epoch 3/50
12373/12381 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.0026 - loss: 0.0574
Epoch 3: val_loss improved from 0.05820 to 0.05533, saving model to best_autoencoder.keras
12381/12381 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - accuracy: 0.0026 - loss: 0.0574 - val_accuracy: 0.0045 - val_loss: 0.0553 - learning_rate: 0.0100
Epoch 4/50
12361/12381 

#### We calculate the reconstruction error for normal test data. Aslo adjust the default threshold to your desired percentile of the normal data's reconstruction error to see what effect it will have.

In [ ]:
# Calculate reconstruction error on test data

reconstructed_data = autoencoder.predict(X_test_scaled)
mse_normal = np.mean(np.power(X_test_scaled - reconstructed_data, 2), axis=1)

# Set threshold for anomaly detection
threshold = np.percentile(mse_normal, 92) #you can adjust this percentile value

#### We apply the autoencoder to the anomalous data and compute its reconstruction error, and combine the reconstruction errors from normal and anomalous data.

In [ ]:
# Test the autoencoder on anomalous data
X_anomalous = df_anomalous.drop(columns=['label'])
X_anomalous_scaled = scaler.transform(X_anomalous)
reconstructed_anomalous = autoencoder.predict(X_anomalous_scaled)
mse_anomalous = np.mean(np.power(X_anomalous_scaled - reconstructed_anomalous, 2), axis=1)

# Combine normal and anomalous MSEs
combined_mse = np.concatenate([mse_normal, mse_anomalous])
combined_labels = np.concatenate([np.zeros(len(mse_normal)), np.ones(len(mse_anomalous))])


#### We classify the data as normal or anomalous based on the threshold and  calculate precision, recall, and F1-score to evaluate the model.

In [ ]:

# Evaluate the model
y_pred = [1 if e > threshold else 0 for e in combined_mse]

precision = precision_score(combined_labels, y_pred)
recall = recall_score(combined_labels, y_pred)
f1 = f1_score(combined_labels, y_pred)

print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")

#### plot the training and validation loss to observe how the model performed over the epochs.

In [ ]:
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Training and Validation Loss Over Epochs")
plt.show()

#### Visualize the distribution of reconstruction errors and the anomaly detection threshold.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(combined_mse, bins=50, color='blue', alpha=0.7, label='Reconstruction Error')
plt.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold: {threshold:.6f}')
plt.title("Reconstruction Error Histogram")
plt.xlabel("MSE")
plt.ylabel("Frequency")
plt.legend()
plt.show()

#### plot the reconstruction error for anomalous data only, showing how well the autoencoder reconstructs normal samples.

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(mse_anomalous, bins=50, color='red', alpha=0.7, label='Anomalous Data')
plt.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold: {threshold:.4f}')
plt.title("Reconstruction Error Histogram for Anomalous Data")
plt.xlabel("MSE")
plt.ylabel("Frequency")
plt.legend()
plt.show()